In [1]:
# Installs and  imports
# pip install -q implicit
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz, load_npz
import implicit
import os
import json
import time

### Configuration

In [4]:
USER_COL = "user"
ITEM_COL = "isbn"
RATING_COL = "rating"

MIN_USER_RATINGS = 5
MIN_ITEM_RATINGS = 5
MAX_USER_RATINGS = 300     # cap extreme power users
IMPLICIT_THRESHOLDS = [(8, 3.0), (6, 2.0), (4, 1.0)]  # rating→weight mapping
CLIP_TO_BINARY = False

USE_BM25 = True
BM25_K1 = 1.2
BM25_B  = 0.75

ALS_FACTORS = 128
ALS_REG     = 0.05
ALS_ITERS   = 20
ALS_USE_CG  = True
ALS_ALPHA   = 60.0

TOPK = 10
SAMPLED_NEG_K = 1000

### Load and preprocess

In [7]:
ratings_prepared_data = pd.read_csv("data/df_ratings_prepared.csv")
df = ratings_prepared_data[[USER_COL, ITEM_COL, RATING_COL]].copy()
df[USER_COL] = df[USER_COL].astype(str)
df[ITEM_COL] = df[ITEM_COL].astype(str)
df[RATING_COL] = pd.to_numeric(df[RATING_COL], errors="coerce")
df = df.dropna(subset=[RATING_COL])

# implicit weight mapping
def to_implicit_weight(x):
    for thr, w in IMPLICIT_THRESHOLDS:
        if x >= thr:
            return w
    return 0.0
df["val"] = df[RATING_COL].apply(to_implicit_weight).astype(np.float32)
df = df[df["val"] > 0]

# cap power users
def cap_user(dfu):
    if len(dfu) <= MAX_USER_RATINGS:
        return dfu
    return dfu.nlargest(MAX_USER_RATINGS, ["val", RATING_COL])
df = df.groupby(USER_COL, group_keys=False).apply(cap_user)

# iterative prune
def iterative_prune(x, min_u, min_i):
    last = -1
    while last != len(x):
        last = len(x)
        uc = x.groupby(USER_COL)[ITEM_COL].count()
        ic = x.groupby(ITEM_COL)[USER_COL].count()
        keep_u = uc[uc >= min_u].index
        keep_i = ic[ic >= min_i].index
        x = x[x[USER_COL].isin(keep_u) & x[ITEM_COL].isin(keep_i)]
    return x

df = iterative_prune(df, MIN_USER_RATINGS, MIN_ITEM_RATINGS)

# map to ids
ucat = pd.Categorical(df[USER_COL]); icat = pd.Categorical(df[ITEM_COL])
df["uid"] = ucat.codes.astype(np.int32)
df["iid"] = icat.codes.astype(np.int32)
uid2raw = pd.Series(ucat.categories, index=np.arange(len(ucat.categories)))
iid2raw = pd.Series(icat.categories, index=np.arange(len(icat.categories)))
n_users = int(df["uid"].max()) + 1
n_items = int(df["iid"].max()) + 1
print("Final dataset:", n_users, "users ×", n_items, "items, interactions=", len(df))

Final dataset: 6365 users × 7791 items, interactions= 100577


### Train/test split

In [9]:
def train_test_split_per_user(df_ui, holdout_per_user=1, seed=123):
    rng = np.random.default_rng(seed)
    x = df_ui.copy()
    x["_r"] = rng.random(len(x))
    x = x.sort_values(["uid", "_r"])
    x["idx"] = x.groupby("uid").cumcount()
    sizes = x.groupby("uid")["idx"].transform("max") + 1
    x["is_test"] = x["idx"] >= (sizes - holdout_per_user)
    train = x[~x["is_test"]][["uid","iid","val"]]
    test  = x[ x["is_test"]][["uid","iid","val"]]
    # drop users with <2 train interactions
    has2 = train.groupby("uid")["iid"].count()
    keep_u = has2[has2 >= 2].index
    return train[train["uid"].isin(keep_u)], test[test["uid"].isin(keep_u)]

train_df, test_df = train_test_split_per_user(df, holdout_per_user=1)
n_users_tr = int(train_df["uid"].max()) + 1
n_items_tr = int(train_df["iid"].max()) + 1

### Build train matrix + weighting

In [11]:
def build_R_ui(train_df, n_users, n_items, clip_binary=False):
    rows = train_df["uid"].to_numpy()
    cols = train_df["iid"].to_numpy()
    vals = train_df["val"].astype(np.float32).to_numpy()
    if clip_binary:
        vals = (vals > 0).astype(np.float32)
    return csr_matrix((vals, (rows, cols)), shape=(n_users, n_items), dtype=np.float32)

R_ui = build_R_ui(train_df, n_users_tr, n_items_tr, clip_binary=CLIP_TO_BINARY)

if USE_BM25:
    R_weighted = implicit.nearest_neighbours.bm25_weight(R_ui.T, K1=BM25_K1, B=BM25_B).T
else:
    R_weighted = implicit.nearest_neighbours.tfidf_weight(R_ui.T).T

C_iu = (R_weighted * ALS_ALPHA).T.tocsr()  # ITEM×USER

### Train ALS with factor swap

In [13]:
def train_als(C_item_user, factors=128, reg=0.05, iterations=20, use_cg=True, seed=42):
    model = implicit.als.AlternatingLeastSquares(
        factors=factors, regularization=reg, iterations=iterations,
        use_cg=use_cg, random_state=seed
    )
    model.fit(C_item_user, show_progress=False)

    # After fitting on ITEM×USER:
    items_f = model.user_factors.copy()   # (n_items, k)
    users_f = model.item_factors.copy()   # (n_users, k)

    # Swap to correct orientation
    model.user_factors = users_f
    model.item_factors = items_f

    return model, users_f, items_f

als_model, U_users, V_items = train_als(C_iu, ALS_FACTORS, ALS_REG, ALS_ITERS, ALS_USE_CG)

C:\Users\Lenovo\anaconda3\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: Intel MKL BLAS is configured to use 10 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


### Align artifacts

In [15]:
R_train = C_iu.T.tocsr()  # users×items
N_USERS, N_ITEMS = R_train.shape
seen = [set(R_train[u].indices.tolist()) for u in range(N_USERS)]
item_pop = np.asarray(R_train.getnnz(axis=0)).astype(np.int32)
pop_rank = np.argsort(-item_pop)

assert U_users.shape[0] == N_USERS
assert V_items.shape[0] == N_ITEMS

### Recommender

In [17]:
def als_recommend_for_user(uid, topn=10, blend_pop=0.0):
    uid = int(uid)
    if uid < 0 or uid >= N_USERS or R_train[uid].nnz == 0:
        return []
    user_row = R_train[uid]
    filt = np.fromiter((i for i in seen[uid]), dtype=np.int32) if seen[uid] else np.empty(0, np.int32)

    item_ids, scores = als_model.recommend(
        userid=uid, user_items=user_row, N=max(topn*2, topn),
        filter_items=filt, filter_already_liked_items=False, recalculate_user=False
    )

    if blend_pop <= 0:
        return [(int(i), float(s)) for i, s in zip(item_ids[:topn], scores[:topn])]

    s = np.asarray(scores, dtype=np.float32)
    p = item_pop[item_ids].astype(np.float32)
    mm = lambda x: np.zeros_like(x) if x.max() <= x.min() else (x - x.min())/(x.max()-x.min())
    final = (1-blend_pop)*mm(s) + blend_pop*mm(p)
    order = np.argsort(-final)[:topn]
    return [(int(item_ids[i]), float(final[i])) for i in order]

def popularity_recommend_for_user(uid, topn=10):
    already = seen[uid]
    out = []
    for i in pop_rank:
        if i not in already:
            out.append((int(i), float(item_pop[i])))
            if len(out) >= topn: break
    return out


### Evaluation

In [19]:
gt = test_df.groupby("uid")["iid"].apply(set)
gt = gt[[0 <= u < N_USERS and R_train[u].nnz > 0 for u in gt.index]]

def recall_at_k(pred, truth, k=10):
    return len(set(pred[:k]) & truth)/len(truth) if truth else 0.0

def apk(pred, truth, k=10):
    if not truth: return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(pred[:k], 1):
        if p in truth:
            hits += 1; score += hits/i
    return score/min(len(truth), k)

def evaluate_all_items(gt_dict, topn=10, rec_fn=als_recommend_for_user):
    recalls, maps = [], []
    for uid in gt_dict.index:
        preds = [i for i,_ in rec_fn(uid, topn=topn)]
        truth = gt_dict[uid]
        recalls.append(recall_at_k(preds, truth, k=topn))
        maps.append(apk(preds, truth, k=topn))
    return np.mean(recalls), np.mean(maps)

def evaluate_sampled(gt_dict, topn=10, negatives=1000):
    rng = np.random.default_rng(42)
    recalls, maps = [], []
    for uid in gt_dict.index:
        truth = list(gt_dict[uid])
        if not truth: continue
        banned = seen[uid]
        cand = set(truth)
        while len(cand) < len(truth) + negatives:
            j = int(rng.integers(0, N_ITEMS))
            if j not in banned: cand.add(j)
        cand = np.fromiter(cand, dtype=np.int32)

        uvec = U_users[uid]
        scores = V_items[cand] @ uvec
        order = np.argsort(-scores)[:topn]
        preds = cand[order].tolist()

        recalls.append(recall_at_k(preds, set(truth), k=topn))
        maps.append(apk(preds, set(truth), k=topn))
    return np.mean(recalls), np.mean(maps)

als_r10, als_m10 = evaluate_all_items(gt, topn=TOPK)
pop_r10, pop_m10 = evaluate_all_items(gt, topn=TOPK, rec_fn=popularity_recommend_for_user)
samp_r10, samp_m10 = evaluate_sampled(gt, topn=TOPK, negatives=SAMPLED_NEG_K)
print("ALS:", als_r10, als_m10)
print("POP:", pop_r10, pop_m10)
print("SAMPLED:", samp_r10, samp_m10)

ALS: 0.07855459544383346 0.037184877616927896
POP: 0.02717989002356638 0.009776555817404206
SAMPLED: 0.18067556952081698 0.0950563598049851


### Saving the first Version

In [32]:
MODEL_DIR = "models/als_v0"
os.makedirs(MODEL_DIR, exist_ok=True)
np.save(f"{MODEL_DIR}/U_users.npy", U_users.astype(np.float32))
np.save(f"{MODEL_DIR}/V_items.npy", V_items.astype(np.float32))
save_npz(f"{MODEL_DIR}/R_train.npz", R_train.astype(np.float32))
uid2raw.to_frame("user_raw").to_csv(f"{MODEL_DIR}/uid2raw.csv", index_label="uid")
iid2raw.to_frame("isbn_raw").to_csv(f"{MODEL_DIR}/iid2raw.csv", index_label="iid")
with open(f"{MODEL_DIR}/meta.json","w") as f:
    json.dump({"factors": int(U_users.shape[1]), "topk": TOPK}, f)

# Load later
U_users = np.load(f"{MODEL_DIR}/U_users.npy")
V_items = np.load(f"{MODEL_DIR}/V_items.npy")
R_train = load_npz(f"{MODEL_DIR}/R_train.npz").tocsr()
uid2raw = pd.read_csv(f"{MODEL_DIR}/uid2raw.csv", index_col="uid")["user_raw"]
iid2raw = pd.read_csv(f"{MODEL_DIR}/iid2raw.csv", index_col="iid")["isbn_raw"]

als_loaded = implicit.als.AlternatingLeastSquares(factors=U_users.shape[1], iterations=1, use_cg=True)
als_loaded.user_factors = U_users.copy()
als_loaded.item_factors = V_items.copy()


### Adjusting Hyperparameters

In [35]:
# Narrow ranges around current good values
grid = {
    "factors": [128, 192, 256],
    "reg":     [0.04, 0.06, 0.08],
    "alpha":   [50.0, 70.0, 90.0],
    "iters":   [20, 28],
    "blend":   [0.00, 0.05, 0.10],   # optional popularity blend at inference
}

results = []
run_id = 0

def fit_and_eval(R_weighted, alpha, factors, reg, iters, blend):
    """
    Fits ALS on C_iu = alpha * R_weighted^T, swaps factors to (users, items),
    rebuilds aligned artifacts, then evaluates with your existing eval functions.
    Returns (als_r10, als_m10, samp_r10, samp_m10, train_time_s).
    """
    global als_model, U_users, V_items, R_train, N_USERS, N_ITEMS, seen, item_pop, pop_rank

    t0 = time.time()

    # Build confidence (ITEM×USER)
    C_try = (R_weighted * alpha).T.tocsr()

    # Train ALS
    model = implicit.als.AlternatingLeastSquares(
        factors=factors, regularization=reg, iterations=iters, use_cg=True, random_state=42
    )
    model.fit(C_try, show_progress=False)

    # Swap factors so .recommend() expects users in user_factors, items in item_factors
    items_f = model.user_factors.copy()   # (n_items, k)
    users_f = model.item_factors.copy()   # (n_users, k)
    model.user_factors = users_f
    model.item_factors = items_f

    # Bind globals for recommender/eval functions
    als_model = model
    U_users = users_f
    V_items = items_f
    R_train = C_try.T.tocsr()  # users×items
    N_USERS, N_ITEMS = R_train.shape
    seen = [set(R_train[u].indices.tolist()) for u in range(N_USERS)]
    item_pop = np.asarray(R_train.getnnz(axis=0)).astype(np.int32)
    pop_rank = np.argsort(-item_pop)

    # Local wrapper to pass blend
    def _als_rec_fn(uid, topn=TOPK):
        return als_recommend_for_user(uid, topn=topn, blend_pop=blend)

    # Evaluate
    als_r10, als_m10 = evaluate_all_items(gt, topn=TOPK, rec_fn=_als_rec_fn)

    # Sampled negatives via direct dot-product (independent of .recommend())
    rng = np.random.default_rng(42)
    recalls, maps = [], []
    for uid in gt.index:
        truth = list(gt[uid])
        if not truth: 
            continue
        banned = seen[uid]
        cand = set(truth)
        while len(cand) < len(truth) + SAMPLED_NEG_K:
            j = int(rng.integers(0, N_ITEMS))
            if j not in banned:
                cand.add(j)
        cand = np.fromiter(cand, dtype=np.int32)

        uvec = U_users[uid]                 # (k,)
        scores = V_items[cand] @ uvec       # (|cand|,)
        order = np.argsort(-scores)[:TOPK]
        preds = cand[order].tolist()

        recalls.append(recall_at_k(preds, set(truth), k=TOPK))
        maps.append(apk(preds, set(truth), k=TOPK))
    samp_r10, samp_m10 = float(np.mean(recalls)), float(np.mean(maps))

    t1 = time.time()
    return als_r10, als_m10, samp_r10, samp_m10, (t1 - t0)

# ---- Run the sweep
for k in grid["factors"]:
    for reg in grid["reg"]:
        for a in grid["alpha"]:
            for it in grid["iters"]:
                for blend in grid["blend"]:
                    run_id += 1
                    r_all, m_all, r_samp, m_samp, secs = fit_and_eval(R_weighted, a, k, reg, it, blend)
                    results.append((run_id, k, reg, a, it, blend, r_all, m_all, r_samp, m_samp, secs))
                    print(f"[{run_id:03d}] f={k:3} reg={reg:.3f} α={a:5.1f} it={it:2d} "
                          f"blend={blend:0.2f}  "
                          f"ALS R@{TOPK}={r_all:.4f} MAP@{TOPK}={m_all:.4f}  "
                          f"SAMP R@{TOPK}={r_samp:.4f} MAP@{TOPK}={m_samp:.4f}  "
                          f"time={secs:.1f}s")

# ---- Results table
res_cols = ["run","factors","reg","alpha","iters","blend_pop",
            f"ALS_R@{TOPK}", f"ALS_MAP@{TOPK}",
            f"SAMP_R@{TOPK}", f"SAMP_MAP@{TOPK}", "train_secs"]
res_df = pd.DataFrame(results, columns=res_cols)

# Top-10 by ALS Recall then MAP
display(res_df.sort_values([f"ALS_R@{TOPK}", f"ALS_MAP@{TOPK}"], ascending=[False, False]).head(10))

# Also show top-10 by ALS MAP if you care more about ranking quality
display(res_df.sort_values([f"ALS_MAP@{TOPK}", f"ALS_R@{TOPK}"], ascending=[False, False]).head(10))


[001] f=128 reg=0.040 α= 50.0 it=20 blend=0.00  ALS R@10=0.0792 MAP@10=0.0379  SAMP R@10=0.1818 MAP@10=0.0959  time=80.4s
[002] f=128 reg=0.040 α= 50.0 it=20 blend=0.05  ALS R@10=0.0801 MAP@10=0.0379  SAMP R@10=0.1818 MAP@10=0.0959  time=83.5s
[003] f=128 reg=0.040 α= 50.0 it=20 blend=0.10  ALS R@10=0.0804 MAP@10=0.0379  SAMP R@10=0.1818 MAP@10=0.0959  time=84.9s
[004] f=128 reg=0.040 α= 50.0 it=28 blend=0.00  ALS R@10=0.0798 MAP@10=0.0382  SAMP R@10=0.1860 MAP@10=0.0957  time=85.8s
[005] f=128 reg=0.040 α= 50.0 it=28 blend=0.05  ALS R@10=0.0803 MAP@10=0.0380  SAMP R@10=0.1860 MAP@10=0.0957  time=83.2s
[006] f=128 reg=0.040 α= 50.0 it=28 blend=0.10  ALS R@10=0.0801 MAP@10=0.0376  SAMP R@10=0.1860 MAP@10=0.0957  time=83.1s
[007] f=128 reg=0.040 α= 70.0 it=20 blend=0.00  ALS R@10=0.0776 MAP@10=0.0371  SAMP R@10=0.1780 MAP@10=0.0930  time=79.8s
[008] f=128 reg=0.040 α= 70.0 it=20 blend=0.05  ALS R@10=0.0778 MAP@10=0.0372  SAMP R@10=0.1780 MAP@10=0.0930  time=81.0s
[009] f=128 reg=0.040 α=

,run,factors,reg,alpha,iters,blend_pop,ALS_R@10,ALS_MAP@10,SAMP_R@10,SAMP_MAP@10,train_secs
148,149,256,0.08,50.0,28,0.05,0.089866,0.043999,0.182718,0.098602,240.678018
147,148,256,0.08,50.0,28,0.00,0.089238,0.043956,0.182718,0.098602,248.024609
130,131,256,0.06,50.0,28,0.05,0.089238,0.043497,0.182247,0.098485,242.699528
129,130,256,0.06,50.0,28,0.00,0.089081,0.043621,0.182247,0.098485,237.719887
149,150,256,0.08,50.0,28,0.10,0.088767,0.043872,0.182718,0.098602,245.315656
112,113,256,0.04,50.0,28,0.05,0.088452,0.043856,0.184132,0.098892,253.567242
111,112,256,0.04,50.0,28,0.00,0.088138,0.043733,0.184132,0.098892,256.809261
131,132,256,0.06,50.0,28,0.10,0.087981,0.043428,0.182247,0.098485,230.970328
128,129,256,0.06,50.0,20,0.10,0.087510,0.043950,0.180676,0.098027,200.106317
146,147,256,0.08,50.0,20,0.10,0.087353,0.043906,0.180204,0.097980,201.336193


,run,factors,reg,alpha,iters,blend_pop,ALS_R@10,ALS_MAP@10,SAMP_R@10,SAMP_MAP@10,train_secs
108,109,256,0.04,50.0,20,0.00,0.086096,0.044177,0.181304,0.098635,194.702066
110,111,256,0.04,50.0,20,0.10,0.086881,0.044107,0.181304,0.098635,191.407512
109,110,256,0.04,50.0,20,0.05,0.086410,0.044100,0.181304,0.098635,201.824246
148,149,256,0.08,50.0,28,0.05,0.089866,0.043999,0.182718,0.098602,240.678018
147,148,256,0.08,50.0,28,0.00,0.089238,0.043956,0.182718,0.098602,248.024609
128,129,256,0.06,50.0,20,0.10,0.087510,0.043950,0.180676,0.098027,200.106317
146,147,256,0.08,50.0,20,0.10,0.087353,0.043906,0.180204,0.097980,201.336193
149,150,256,0.08,50.0,28,0.10,0.088767,0.043872,0.182718,0.098602,245.315656
112,113,256,0.04,50.0,28,0.05,0.088452,0.043856,0.184132,0.098892,253.567242
145,146,256,0.08,50.0,20,0.05,0.087196,0.043845,0.180204,0.097980,191.270128


### Best overall (all-items) hit:
- `f=256, reg=0.08, α=50, it=28, blend=0.05 → R@10≈0.0899, MAP@10≈0.0440 (runs ~240–250s)`.


#### Very close runner-ups:

- `f=256, reg=0.06, α=50, it=28, blend=0.00/0.05 → R@10≈0.0891–0.0892, MAP@10≈0.0435–0.0436`.

Speed/quality trade-off: `f=192, reg=0.06, α=50, it=28, blend=0.10 → R@10≈0.0845, MAP@10≈0.0421 (~88s)`.

Patterns:

- `α=50` consistently best; `α=90` consistently worse.
- Increasing factors improve performance but slows it down.
- `iters 28` gives small bumps vs 20.
- `blend 0.05–0.10` gives tiny MAP bumps; recall unchanged.

#### Recommendation 

**Quality-first (final):** `f=256, reg=0.08, α=50, it=28, blend=0.05`

**Speed-first variant:** `f=192, reg=0.06, α=50, it=28, blend=0.10`

### Saving the *better* model

In [42]:
# Params 
FINAL_FACTORS = 256
FINAL_REG     = 0.08
FINAL_ALPHA   = 50.0
FINAL_ITERS   = 28
FINAL_BLEND   = 0.05  # used only at inference

# Build confidence and fit
C_final = (R_weighted * FINAL_ALPHA).T.tocsr()  # item×user
final_model = implicit.als.AlternatingLeastSquares(
    factors=FINAL_FACTORS, regularization=FINAL_REG,
    iterations=FINAL_ITERS, use_cg=True, num_threads=os.cpu_count()
)
final_model.fit(C_final, show_progress=False)

# Swap factors so .recommend expects (users, items)
V_items = final_model.user_factors.copy()   # items
U_users = final_model.item_factors.copy()   # users
final_model.user_factors = U_users
final_model.item_factors = V_items

# Rebuild aligned inference artifacts
R_train = C_final.T.tocsr()  # users×items
N_USERS, N_ITEMS = R_train.shape
seen = [set(R_train[u].indices.tolist()) for u in range(N_USERS)]
item_pop = np.asarray(R_train.getnnz(axis=0)).astype(np.int32)
pop_rank = np.argsort(-item_pop)

def recommend_final(uid, topn=10):
    if uid < 0 or uid >= N_USERS or R_train[uid].nnz == 0:
        return []
    row = R_train[uid]
    filt = np.fromiter((i for i in seen[uid]), dtype=np.int32) if seen[uid] else np.empty(0, np.int32)
    ids, sc = final_model.recommend(uid, user_items=row, N=max(topn*2, topn),
                                    filter_items=filt, filter_already_liked_items=False, recalculate_user=False)
    if FINAL_BLEND <= 0:
        return [(int(i), float(s)) for i, s in zip(ids[:topn], sc[:topn])]
    s = np.asarray(sc, dtype=np.float32)
    p = item_pop[ids].astype(np.float32)
    def mm(x): 
        lo, hi = float(x.min()), float(x.max()); 
        return np.zeros_like(x) if hi<=lo else (x-lo)/(hi-lo)
    final = (1-FINAL_BLEND)*mm(s) + FINAL_BLEND*mm(p)
    order = np.argsort(-final)[:topn]
    return [(int(ids[i]), float(final[i])) for i in order]

# Save everything
MODEL_DIR = "models/als_best"
os.makedirs(MODEL_DIR, exist_ok=True)
np.save(f"{MODEL_DIR}/U_users.npy", U_users.astype(np.float32))
np.save(f"{MODEL_DIR}/V_items.npy", V_items.astype(np.float32))
save_npz(f"{MODEL_DIR}/R_train.npz", R_train.astype(np.float32))
uid2raw.to_frame("user_raw").to_csv(f"{MODEL_DIR}/uid2raw.csv", index_label="uid")
iid2raw.to_frame("isbn_raw").to_csv(f"{MODEL_DIR}/iid2raw.csv", index_label="iid")
with open(f"{MODEL_DIR}/meta.json","w") as f:
    json.dump({
        "factors": FINAL_FACTORS, "reg": FINAL_REG, "alpha": FINAL_ALPHA,
        "iters": FINAL_ITERS, "blend": FINAL_BLEND, "topk": TOPK
    }, f, indent=2)
print("Saved to", MODEL_DIR)


Saved to models/als_best
